In [1]:
import pandas as pd
import requests
import os
import urllib3
import json
import re

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# 1. 데이터 로드 및 저장 설정
df = pd.read_csv('s2648.csv')
save_dir = 'pdb_files_s2648'
os.makedirs(save_dir, exist_ok=True)

unique_pdbs = df['PDB'].unique()
mapping_results = {}

def download_structure(pdb_id):
    pdb_id = pdb_id.lower()
    # PDB 시도
    pdb_url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    pdb_path = os.path.join(save_dir, f"{pdb_id}.pdb")
    
    if os.path.exists(pdb_path): return pdb_path, "PDB"

    try:
        r = requests.get(pdb_url, verify=False, timeout=10)
        if r.status_code == 200:
            with open(pdb_path, 'wb') as f: f.write(r.content)
            return pdb_path, "PDB"
    except: pass

    # CIF 시도
    cif_url = f"https://files.rcsb.org/download/{pdb_id}.cif"
    cif_path = os.path.join(save_dir, f"{pdb_id}.cif")
    try:
        r = requests.get(cif_url, verify=False, timeout=10)
        if r.status_code == 200:
            with open(cif_path, 'wb') as f: f.write(r.content)
            return cif_path, "CIF"
    except: pass
    return None, None

# 2. 실행 루프
print(f"총 {len(unique_pdbs)}개의 PDB 수집 시작...")
for i, pdb_id in enumerate(unique_pdbs, 1):
    path, f_type = download_structure(pdb_id)
    if path:
        # CSV에 체인 정보가 있으므로 이를 활용해 매핑 결과 생성
        # 동일 PDB라도 행마다 체인이 다를 수 있으므로 여기서는 PDB ID 기준으로만 기록
        mapping_results[pdb_id] = {"pdb_id": pdb_id, "file": path, "type": f_type}
        if i % 20 == 0: print(f"[{i}/{len(unique_pdbs)}] 완료...")
    else:
        print(f"❌ 실패: {pdb_id}")

with open('mapping_results_s2648.json', 'w') as f:
    json.dump(mapping_results, f, indent=4)
print("수집 및 JSON 저장 완료!")

총 131개의 PDB 수집 시작...
[20/131] 완료...
[40/131] 완료...
[60/131] 완료...
[80/131] 완료...
[100/131] 완료...
[120/131] 완료...
수집 및 JSON 저장 완료!


In [2]:
import pandas as pd
import json
import os
import re
from Bio.PDB import PDBParser, MMCIFParser, Polypeptide

# 1. 데이터 및 JSON 로드
with open('mapping_results_s2648.json', 'r') as f:
    mapping_results = json.load(f)

# s2648.csv 로드 (CLID, PDB, CHAIN, MUT, DDG 등 포함)
df = pd.read_csv('s2648.csv')

# 2. MUT 컬럼 분리 함수 (C30S -> WT:C, POS:30, MT:S)
def split_mut(mut):
    match = re.match(r"([A-Z])(\d+)([A-Z])", str(mut))
    if match:
        return match.groups()
    return None, None, None

df[['WT', 'POS', 'MT']] = df['MUT'].apply(lambda x: pd.Series(split_mut(x)))
df['POS'] = df['POS'].astype(int)

# 결과 기록용 컬럼 추가
df['PDB_WT_CHECK'] = None
df['STATUS'] = "Missing_File"

print("S2648 인덱스 검증 및 일치성 체크 시작...")

# 3. 매핑 검증 루프
for idx, row in df.iterrows():
    pdb_id = row['PDB']
    chain_id = row['CHAIN']
    target_pos = row['POS']
    target_wt = row['WT']
    
    if pdb_id not in mapping_results:
        continue
        
    info = mapping_results[pdb_id]
    pdb_path = info['file']
    
    # 파일 형식에 따른 파서 선택
    if info['type'] == "CIF":
        parser = MMCIFParser(QUIET=True)
    else:
        parser = PDBParser(QUIET=True)
        
    try:
        struct = parser.get_structure('temp', pdb_path)
        model = struct[0]
        
        # 체인 존재 여부 확인
        if chain_id not in model:
            df.at[idx, 'STATUS'] = "Missing_Chain"
            continue
            
        chain = model[chain_id]
        
        # 해당 위치(POS) 잔기 추출
        if target_pos in chain:
            res = chain[target_pos]
            if Polypeptide.is_aa(res):
                # 3글자 아미노산을 1글자로 변환 (예: CYS -> C)
                pdb_wt = Polypeptide.three_to_one(res.get_resname())
                df.at[idx, 'PDB_WT_CHECK'] = pdb_wt
                
                # 데이터셋의 WT와 일치하는지 비교
                if pdb_wt == target_wt:
                    df.at[idx, 'STATUS'] = "Success"
                else:
                    df.at[idx, 'STATUS'] = "WT_Mismatch"
            else:
                df.at[idx, 'STATUS'] = "Not_Amino_Acid"
        else:
            df.at[idx, 'STATUS'] = "Missing_Residue"
            
    except Exception as e:
        df.at[idx, 'STATUS'] = f"Error: {str(e)}"

# 4. 결과 요약 및 불일치 개수 출력
print("\n" + "="*30)
print("S2648 매핑 검증 요약:")
status_counts = df['STATUS'].value_counts()
print(status_counts)
print("="*30)

success_count = (df['STATUS'] == "Success").sum()
fail_count = len(df) - success_count
print(f"총 데이터 수: {len(df)}")
print(f"일치(Success): {success_count}")
print(f"불일치/오류(Total Fail): {fail_count}")

# 상세 결과 저장
df.to_csv("s2648_mapping_validation.tsv", sep='\t', index=False)
print(f"\n[완료] 상세 매핑 결과가 's2648_mapping_validation.tsv'에 저장되었습니다.")

S2648 인덱스 검증 및 일치성 체크 시작...

S2648 매핑 검증 요약:
STATUS
Success        2644
WT_Mismatch       4
Name: count, dtype: int64
총 데이터 수: 2648
일치(Success): 2644
불일치/오류(Total Fail): 4

[완료] 상세 매핑 결과가 's2648_mapping_validation.tsv'에 저장되었습니다.


In [3]:
import pandas as pd
import re
from collections import Counter

# 1. 파일 로드 (사용자 환경의 경로에 맞춰 수정)
s2450_path = r'C:\Users\Kunny\Documents\GitHub\CAGI\EvoStructCLIP\Stability\S2450.tsv'
s2648_val_path = r'C:\Users\Kunny\Documents\GitHub\CAGI\EvoStructCLIP\Stability\s2648_mapping_validation.tsv'

df_2450 = pd.read_csv(s2450_path, sep='\t')
df_2648 = pd.read_csv(s2648_val_path, sep='\t')

# 2. 전처리: WT_Mismatch 제거 및 로그
mismatch_rows = df_2648[df_2648['STATUS'] == 'WT_Mismatch']
if not mismatch_rows.empty:
    print(f"--- [LOG] WT_Mismatch 데이터 {len(mismatch_rows)}건 제외 ---")
    for _, row in mismatch_rows.iterrows():
        print(f"제외됨: {row['PDB']} {row['MUT']} (DDG: {row['DDG']})")

df_2648_clean = df_2648[df_2648['STATUS'] == 'Success'].copy()

# 매칭을 위해 DDG 반올림 및 키 생성
df_2450['match_key'] = df_2450.apply(lambda r: f"{r['WT']}{r['MT']}_{round(r['DDG'], 2)}", axis=1)
df_2648_clean['match_key'] = df_2648_clean.apply(lambda r: f"{r['WT']}{r['MT']}_{round(r['DDG'], 2)}", axis=1)

# 결과 컬럼 생성
df_2450['MAPPED_PDB'] = None
df_2450['MAPPED_CHAIN'] = None
df_2450['MAPPED_PDB_POS'] = None
df_2450['MAPPED_OFFSET'] = None

# 3. UniProt ID별로 최적의 PDB 매핑 찾기
print("\nUniProt-PDB 매핑 및 Offset 보정 시작...")

for uniprot_id in df_2450['PDB'].unique():
    subset_2450 = df_2450[df_2450['PDB'] == uniprot_id]
    
    # 해당 UniProt의 변이들이 s2648의 어떤 PDB/Offset과 가장 잘 맞는지 후보 수집
    candidates = []
    for _, t_row in subset_2450.iterrows():
        # (WT, MT, DDG)가 일치하는 s2648의 모든 행 찾기
        matches = df_2648_clean[df_2648_clean['match_key'] == t_row['match_key']]
        for _, c_row in matches.iterrows():
            offset = t_row['POS'] - c_row['POS']
            # (PDB_ID, CHAIN, OFFSET) 쌍을 후보로 저장
            candidates.append((c_row['PDB'], c_row['CHAIN'], offset))
            
    if not candidates:
        print(f"⚠️ 매칭 후보 없음: {uniprot_id}")
        continue
        
    # 가장 많이 등장한 (PDB, Chain, Offset) 조합을 정답으로 선택
    best_match = Counter(candidates).most_common(1)[0][0]
    best_pdb, best_chain, best_offset = best_match
    
    # 4. 결정된 정답으로 해당 UniProt의 모든 행 업데이트
    for idx in subset_2450.index:
        t_row = df_2450.loc[idx]
        
        # 선정한 PDB/Chain/Offset 조건에 딱 맞는 행을 s2648에서 최종 선택
        final_row = df_2648_clean[
            (df_2648_clean['match_key'] == t_row['match_key']) & 
            (df_2648_clean['PDB'] == best_pdb) &
            (df_2648_clean['CHAIN'] == best_chain) &
            ((t_row['POS'] - df_2648_clean['POS']) == best_offset)
        ]
        
        if not final_row.empty:
            df_2450.at[idx, 'MAPPED_PDB'] = best_pdb
            df_2450.at[idx, 'MAPPED_CHAIN'] = best_chain
            df_2450.at[idx, 'MAPPED_PDB_POS'] = final_row.iloc[0]['POS']
            df_2450.at[idx, 'MAPPED_OFFSET'] = best_offset
        else:
            # 개별 행이 전체 경향성(Offset)에서 벗어나는 경우
            pass

# 5. 최종 정제 및 저장
# PDB 매핑에 실패한 행(가망 없는 데이터)은 제거
final_df = df_2450.dropna(subset=['MAPPED_PDB']).copy()
final_df = final_df.drop(columns=['match_key'])

print(f"\n--- 매핑 완료 ---")
print(f"원본 S2450: {len(df_2450)}행")
print(f"정제 후 S2450: {len(final_df)}행 (매핑 성공)")

final_df.to_csv("S2450_Final_PDB_Mapped.tsv", sep='\t', index=False)

--- [LOG] WT_Mismatch 데이터 4건 제외 ---
제외됨: 1LVE L27CN (DDG: -0.57)
제외됨: 1LVE L27CQ (DDG: 0.63)
제외됨: 1LVE V27BL (DDG: 1.67)
제외됨: 1LVE Y27DD (DDG: 2.19)

UniProt-PDB 매핑 및 Offset 보정 시작...

--- 매핑 완료 ---
원본 S2450: 2450행
정제 후 S2450: 2398행 (매핑 성공)
